<a href="https://colab.research.google.com/github/sipy/mdRwd/blob/master/1_B1_Particle_Crystallization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F
import math
import time

# ==========================================
# 1. THE GOLDILOCKS HARDWARE SETUP
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float64) # Maximum Precision

print(f"Hardware acquired: {device.type.upper()} (FLOAT64 / 128^3 GRID)")

N = 128    # Massive spatial volume
dx = 0.08  # Tighter pixel density to catch the micro-curvature!

mask = torch.zeros(1, 4, N, N, N, device=device)
mask[:, :, 2:-2, 2:-2, 2:-2] = 1.0
vacuum = torch.tensor([1.0, 0.0, 0.0, 0.0], device=device).view(1, 4, 1, 1, 1)

# ==========================================
# 2. THE TOPOLOGICAL SEED
# ==========================================
print("Generating the B=1 Hedgehog Seed...")

X, Y, Z = torch.meshgrid(torch.arange(N, device=device),
                         torch.arange(N, device=device),
                         torch.arange(N, device=device), indexing='ij')

cx, cy, cz = N // 2, N // 2, N // 2
rx = (X - cx) * dx
ry = (Y - cy) * dx
rz = (Z - cz) * dx
r = torch.sqrt(rx**2 + ry**2 + rz**2) + 1e-8

R_knot = 1.0
f = math.pi * torch.exp(-r / R_knot)

phi0 = torch.cos(f)
phi1 = torch.sin(f) * (rx / r)
phi2 = torch.sin(f) * (ry / r)
phi3 = torch.sin(f) * (rz / r)

phi = torch.stack([phi0, phi1, phi2, phi3], dim=0).unsqueeze(0)
phi = F.normalize(phi, p=2, dim=1)
phi = phi * mask + vacuum * (1.0 - mask)
phi.requires_grad_(True)

# ==========================================
# 3. TUNED 4TH-ORDER PHYSICS ENGINE
# ==========================================
def get_derivatives_4th_order(phi_tensor, dx_val):
    k_vals = [1.0/12.0, -8.0/12.0, 0.0, 8.0/12.0, -1.0/12.0]
    kernel = torch.tensor(k_vals, device=device) / dx_val

    kx = kernel.view(1, 1, 1, 1, 5).repeat(4, 1, 1, 1, 1)
    ky = kernel.view(1, 1, 1, 5, 1).repeat(4, 1, 1, 1, 1)
    kz = kernel.view(1, 1, 5, 1, 1).repeat(4, 1, 1, 1, 1)

    pad_x = F.pad(phi_tensor, (2, 2, 0, 0, 0, 0), mode='replicate')
    dx_phi = F.conv3d(pad_x, kx, groups=4)

    pad_y = F.pad(phi_tensor, (0, 0, 2, 2, 0, 0), mode='replicate')
    dy_phi = F.conv3d(pad_y, ky, groups=4)

    pad_z = F.pad(phi_tensor, (0, 0, 0, 0, 2, 2), mode='replicate')
    dz_phi = F.conv3d(pad_z, kz, groups=4)

    return torch.stack([dx_phi, dy_phi, dz_phi], dim=0)

# THE GOLDILOCKS PARAMETERS
# e = 1.0 (Standard Skyrme Repulsion)
# m_pi = 0.15 (Gentle Chiral Mass to contain the tails without crushing the core)
def compute_skyrme_energy(phi_tensor, dx_val, f_pi=1.0, e=1.0, m_pi=0.15):
    dphi = get_derivatives_4th_order(phi_tensor, dx_val)
    G = torch.einsum('iczyx, jczyx -> ijzyx', dphi[:, 0], dphi[:, 0])

    E2 = G[0,0] + G[1,1] + G[2,2]
    TrG_sq = E2**2
    Tr_G2 = sum(G[i,j] * G[j,i] for i in range(3) for j in range(3))
    E4 = 0.5 * (TrG_sq - Tr_G2)

    E_mass = (m_pi**2) * (1.0 - phi_tensor[0, 0])

    energy_density = (f_pi**2 / 16.0) * E2 + (1.0 / (32.0 * e**2)) * E4 + E_mass
    return torch.sum(energy_density) * (dx_val**3)

def compute_winding_number(phi_tensor, dx_val):
    dphi = get_derivatives_4th_order(phi_tensor, dx_val)
    vectors = torch.stack([phi_tensor[0], dphi[0,0], dphi[1,0], dphi[2,0]], dim=0)
    matrix_field = vectors.permute(2, 3, 4, 0, 1)
    det_density = torch.linalg.det(matrix_field)
    return torch.sum(det_density) * (dx_val**3) / (2 * math.pi**2)

# ==========================================
# 4. THE MASTER RELAXATION SEQUENCE
# ==========================================
with torch.no_grad():
    initial_B = compute_winding_number(phi, dx)
    print(f"Initial Seed Winding B: {initial_B.item():.5f}\n")

optimizer = torch.optim.Adam([phi], lr=0.01)
print("Initiating Goldilocks Particle Birth...")
start_time = time.time()

for step in range(1001):
    optimizer.zero_grad()

    energy = compute_skyrme_energy(phi, dx)
    energy.backward()

    # Ultra-strict gradient clipping to prevent mathematical tearing
    phi.grad.clamp_(-0.02, 0.02)
    optimizer.step()

    with torch.no_grad():
        phi.data = phi.data * mask + vacuum * (1.0 - mask)
        phi.copy_(F.normalize(phi, p=2, dim=1))

    if step % 100 == 0:
        with torch.no_grad():
            B = compute_winding_number(phi, dx)
            elapsed = time.time() - start_time
            print(f"Step {step:4d} | Time: {elapsed:.1f}s | Energy: {energy.item():.4f} | Winding B: {B.item():.5f}")

print("\nSimulation Complete. The Particle is born.")

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from skimage import measure

# Safety check for Colab sessions
if 'phi' not in globals():
    print("CRITICAL ERROR: 'phi' not found. Please run the simulation cell above first.")
else:
    print("Extracting B=1 Particle Geometry...")

    with torch.no_grad():
        # In a B=1 Skyrmion, phi_0 goes from 1.0 (vacuum) to -1.0 (center).
        # We plot the 'shell' where phi_0 = 0.0 to see the particle's boundary.
        volume = phi[0, 0, :, :, :].cpu().numpy()

    try:
        # Marching Cubes finds the 'surface' of the particle
        # We use level=0.0 to capture the 'equator' of the topological twist
        verts, faces, normals, values = measure.marching_cubes(volume, level=0.0)

        print("Rendering Fundamental Fermion (B=1)...")

        fig = go.Figure(data=[go.Mesh3d(
            x=verts[:, 0],
            y=verts[:, 1],
            z=verts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            opacity=0.9,
            color='deepskyblue', # High-energy blue for a stable particle
            flatshading=False    # Smooth shading makes the sphere look organic
        )])

        fig.update_layout(
            title='B=1 Topological Particle (Stable Hedgehog)',
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                bgcolor='rgb(15, 15, 15)'
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )

        fig.show()

    except ValueError:
        print("Error: Could not find an isosurface. The particle might have dissolved.")